In [1]:
!nvidia-smi
!pip install transformers datasets torch

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
text_data = """
Artificial Intelligence is transforming industries worldwide.
Machine learning enables computers to learn from data.
Deep learning models require large datasets.
Natural Language Processing helps machines understand text.
GPT models generate human-like text.
"""

with open("data.txt", "w") as f:
    f.write(text_data)

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch

# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Load model
model = GPT2LMHeadModel.from_pretrained("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
dataset = load_dataset("text", data_files={"train": "data.txt"})

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=200,
    logging_steps=50,
    save_total_limit=1,
    report_to="none"   # prevents wandb error in colab
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6, training_loss=4.069780031840007, metrics={'train_runtime': 48.1607, 'train_samples_per_second': 0.249, 'train_steps_per_second': 0.125, 'total_flos': 783876096000.0, 'train_loss': 4.069780031840007, 'epoch': 2.0})

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,          # use trained model directly
    tokenizer=tokenizer
)

prompt = "Artificial Intelligence"

output = generator(
    prompt,
    max_length=50,
    num_return_sequences=1,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

print(output[0]["generated_text"])

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Artificial Intelligence (AI) has been a major focus of research for many years. The AI revolution has seen many breakthroughs, but this is one of the most recent. A machine learning is an emerging field, and many AI researchers are trying to get out of it. AI has many advantages over traditional AI. First, it enables us to predict the future. The most important advantage for AI is that it has a large set of algorithms. This makes it the most important advantage. AI is able to predictors that are computationally accurate. The reason for this is that it is very fast. A machine learning is a large data set and very fast. This is why it is so important for AI. AI can be used to predict the future. A machine learning is a computational approach to learn a human's human's knowledge. It has many advantages over traditional AI.

The AI revolution has made AI much more powerful. In many ways, it has become the most powerful artificial intelligence ever. It has many advantages over traditional A